# Week 20, Day 1: Entropy: measuring uncertainty in bits

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w020/d1.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Compute entropy

Write `entropy(p)` in bits for any array of probabilities, skipping zeros. Then, for the joint table `pxy` of two dependent binary variables, compute `H_joint`, the marginal entropies `H_x` and `H_y`, and check that H(X, Y) is less than H(X) + H(Y).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def entropy(p):
    p = np.asarray(p, dtype=float)
    p = p[p > 0]
    return ...

pxy = np.array([[0.4, 0.1],
                [0.1, 0.4]])            # rows: X, columns: Y
H_joint = entropy(pxy.ravel())
H_x, H_y = ..., ...
print(f"H(X) = {H_x:.3f}, H(Y) = {H_y:.3f}, H(X, Y) = {H_joint:.3f} bits;  H(Y | X) = {H_joint - H_x:.3f}")

In [ ]:
import numpy as np
def test_values():
    "entropy gives 1 bit for a fair coin, 2 bits for four equal outcomes and 0 for a certainty"
    assert abs(entropy([0.5, 0.5]) - 1) < 1e-12 and abs(entropy([0.25] * 4) - 2) < 1e-12 and entropy([1.0, 0.0]) == 0

def test_zero_safe():
    "zero probabilities contribute nothing and cause no warnings or nans"
    assert np.isfinite(entropy([0.0, 0.3, 0.7]))

def test_joint():
    "for the dependent pair, H(X, Y) is less than H(X) + H(Y)"
    assert H_joint < H_x + H_y - 0.05

test_values()
test_zero_safe()
test_joint()
print("All checks passed")

### Exercise: A Huffman code

Write `huffman_lengths(p)` returning the codeword length of each symbol: keep a list of (probability, list of symbol indices) groups, repeatedly merge the two smallest groups and add 1 to the length of every symbol in them. Apply it to a skewed distribution over 8 symbols and compare the average length with the entropy.

In [ ]:
def huffman_lengths(p):
    lengths = [0] * len(p)
    groups = [(pi, [i]) for i, pi in enumerate(p)]
    while len(groups) > 1:
        groups.sort(key=lambda g: g[0])
        (p1, s1), (p2, s2) = groups[0], groups[1]
        for i in s1 + s2:
            ...
        groups = groups[2:] + [...]
    return lengths

letters = np.array([0.35, 0.2, 0.15, 0.1, 0.08, 0.06, 0.04, 0.02])
L = np.array(huffman_lengths(letters))
avg_len = np.sum(letters * L)
H_letters = entropy(letters)
print(f"code lengths {L};  average {avg_len:.3f} bits;  entropy {H_letters:.3f} bits;  a fixed-length code would need 3 bits")

In [ ]:
import numpy as np
def test_lengths():
    "huffman_lengths gives lengths 1, 2, 3, 3 for probabilities 0.5, 0.25, 0.125, 0.125"
    assert sorted(huffman_lengths([0.5, 0.25, 0.125, 0.125])) == [1, 2, 3, 3]

def test_bound():
    "the average Huffman length is within one bit above the entropy"
    assert H_letters <= avg_len < H_letters + 1

test_lengths()
test_bound()
print("All checks passed")

### Exercise: Spike words with and without refractoriness

Write `word_entropy(train, L)` that cuts a binary spike train into non-overlapping words of L bins, counts each word (as an integer), and returns the entropy in bits. Compare an independent-bin neuron firing at 80 Hz (spike probability p per 2 ms bin) and a neuron of the same rate that cannot fire in the 2 bins after a spike, using words of 8 bins from 400,000 bins.

In [ ]:
rng = np.random.default_rng(1948)
T, L = 400000, 8
p = 80 * 0.002                                         # 80 Hz, 2 ms bins

def word_entropy(train, L):
    n = len(train) // L
    words = train[:n * L].reshape(n, L) @ (2 ** np.arange(L))      # each word as an integer
    counts = np.bincount(words, minlength=2 ** L)
    return ...

indep = (rng.random(T) < p).astype(int)
# refractory neuron: after a spike, 2 silent bins; raise the per-bin probability to keep the same rate
q = p / (1 - 2 * p)
refr = np.zeros(T, int); t = 0
while t < T:
    if rng.random() < q:
        refr[t] = 1; t += 3
    else:
        t += 1
h_bin = entropy([p, 1 - p])
H_indep, H_refr = word_entropy(indep, L), word_entropy(refr, L)
print(f"rates: {indep.mean() / 0.002:.1f} and {refr.mean() / 0.002:.1f} Hz;  L h(p) = {L * h_bin:.3f};  word entropy independent {H_indep:.3f}, refractory {H_refr:.3f} bits")

In [ ]:
import numpy as np
def test_word_entropy():
    "word_entropy of independent bins is close to L h(p) (within 0.1 bit)"
    assert abs(H_indep - 8 * h_bin) < 0.1

def test_refractory_lower():
    "the refractory neuron's words have lower entropy than independent bins at the same rate, by more than 0.4 bits"
    assert H_refr < H_indep - 0.4

test_word_entropy()
test_refractory_lower()
print("All checks passed")